In [2]:
import pandas as pd
df = pd.read_csv('/Users/annu/Desktop/Financial Misinformation Detection with LLM Hallucination Auditing + Market Impact Analysis/data/headlines_verified.csv')
def build_prompt(headline):
    return f"""You are a financial fact-checking assistant specializing in Indian and global financial markets.

Your task is to classify the following financial news headline as either FACTUAL or MISLEADING.

- FACTUAL: The headline accurately represents a real financial event, uses precise language, and does not exaggerate or distort facts.
- MISLEADING: The headline exaggerates, distorts, omits critical context, uses sensational language, or is designed to manipulate market sentiment.

Headline: "{headline}"

Respond in this exact format and nothing else:
CLASSIFICATION: <FACTUAL or MISLEADING>
CONFIDENCE: <HIGH or MEDIUM or LOW>
REASONING: <one sentence explaining your decision>"""

In [ ]:

from groq import Groq
import time
from dotenv import load_dotenv
import os
load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

def generate_content(prompt):
    chat = client.chat.completions.create(
      model="llama-3.1-8b-instant",  # smaller = faster response time
        messages=[{"role": "user", "content": prompt}]
    )
    return chat.choices[0].message.content


#(The Request)
response = []

def parse_response(text):
    lines = text.strip().split("\n")
    classification = lines[0].split(':')[1].strip()
    confidence = lines[1].split(':')[1].strip()
    reasoning = lines[2].split(':')[1].strip()

    return {
        'classification' : classification,
        'confidence': confidence,
        'reasoning' : reasoning
    }

for index, row in df.iterrows():
    single_headline = row['headline']
    prompt = build_prompt(single_headline)
    row_response = generate_content(prompt)
    parsed = parse_response(row_response)
    response.append(parsed)
    time.sleep(1)

print(response)






[{'classification': 'MISLEADING', 'confidence': 'MEDIUM', 'reasoning': "While the headline appears to accurately report Hindenburg Research's accusations, its language is sensational and implies guilt, which may not reflect the complexity of the actual allegations or the company's response."}, {'classification': 'FACTUAL', 'confidence': 'HIGH', 'reasoning': "The headline accurately reports on a specific event involving the Adani Group's response to the Hindenburg report, indicating a factual representation of the situation."}, {'classification': 'MISLEADING', 'confidence': 'HIGH', 'reasoning': 'The headline exaggerates the impact by implying "rejected rebuttal" caused the loss, whereas the actual cause of the selloff would require more nuanced analysis of market sentiment and economic factors.'}, {'classification': 'MISLEADING', 'confidence': 'HIGH', 'reasoning': "The headline implies that Adani Enterprises abandoned the share sale due to investor protection when the actual reason is u

In [6]:
!pip3 install python-dotenv

In [4]:
import pandas as pd

results_df = pd.DataFrame(response)
print(results_df.shape)
print(results_df['classification'].value_counts())
print(results_df['confidence'].value_counts())
print(results_df.head())

(87, 3)
classification
FACTUAL       68
MISLEADING    18
MISLEADNG      1
Name: count, dtype: int64
confidence
HIGH      78
MEDIUM     9
Name: count, dtype: int64
  classification confidence                                          reasoning
0     MISLEADING     MEDIUM  While the headline appears to accurately repor...
1        FACTUAL       HIGH  The headline accurately reports on a specific ...
2     MISLEADING       HIGH  The headline exaggerates the impact by implyin...
3     MISLEADING       HIGH  The headline implies that Adani Enterprises ab...
4        FACTUAL       HIGH  The headline accurately reports the intraday s...


In [5]:
results_df = pd.DataFrame(response)
final_df = pd.concat([df.reset_index(drop=True), results_df], axis=1)
final_df.to_csv('headlines_layer1.csv', index=False)
print("Saved!")
print(final_df.shape)
print(final_df['classification'].value_counts())

Saved!
(87, 9)
classification
FACTUAL       68
MISLEADING    18
MISLEADNG      1
Name: count, dtype: int64
